<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_13_mini_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import math

In [2]:
embeddings=nn.Embedding(3,4)

In [3]:
tokens=torch.tensor([0,1,2])

In [4]:
x=embeddings(tokens)

In [5]:
x.shape

torch.Size([3, 4])

In [6]:
x

tensor([[ 1.4053, -1.0579, -1.2416, -0.3483],
        [-0.8898, -1.2856, -1.9578,  0.2163],
        [-0.8779, -0.8154, -2.2485, -1.2978]], grad_fn=<EmbeddingBackward0>)

In [7]:
Wq=nn.Linear(4,4)
Wk=nn.Linear(4,4)
Wv=nn.Linear(4,4)

In [8]:
Wq

Linear(in_features=4, out_features=4, bias=True)

In [9]:
Wk

Linear(in_features=4, out_features=4, bias=True)

In [10]:
Q=Wq(x)
K=Wk(x)
V=Wv(x)

In [11]:
Q

tensor([[-0.2443,  1.3265, -1.7346, -0.1403],
        [ 0.8998,  2.1749, -1.0220, -0.9433],
        [ 1.6586,  1.3904, -1.5287, -1.2410]], grad_fn=<AddmmBackward0>)

In [12]:
K

tensor([[-0.3512,  0.9630, -0.8063, -0.2802],
        [ 1.2520,  1.4807, -1.6354,  0.9141],
        [ 0.5631,  2.1934, -2.2518,  1.6838]], grad_fn=<AddmmBackward0>)

In [13]:
V

tensor([[ 0.9634, -0.4891,  0.5096, -0.2301],
        [ 1.1398, -1.4128, -0.1084, -0.2700],
        [ 0.9969, -0.9550, -0.2723, -0.4313]], grad_fn=<AddmmBackward0>)

In [14]:
scores=Q @ K.T

In [15]:
scores=scores / math.sqrt(4)

In [16]:
weights=torch.softmax(
    scores,
    dim=-1
)

In [17]:
context=weights @ V

In [18]:
print(context)

tensor([[ 1.0267, -1.0123, -0.1505, -0.3721],
        [ 1.0436, -1.0642, -0.1267, -0.3519],
        [ 1.0609, -1.1253, -0.1196, -0.3360]], grad_fn=<MmBackward0>)


**Explain self-attention**

Self-attention allows each token to compare itself with every other token using Query and Key vectors. The resulting attention weights determine how much information to gather from the Value vectors, creating a context-aware representation of each word

# **Day_14 _Multi-Head Attention**

where 8 attention mechanisms run simultaneously, and you'll finally understand why GPT uses many heads instead of one. This is the point where our tiny Transformer starts looking like a real GPT

In [7]:
embed_dim=8
num_heads=2

In [8]:
head_dim = embed_dim // num_heads

print(head_dim)

4


**Real GPT**

GPT-2:

12 heads

GPT-3:

96 heads

Modern models:

100+

In [9]:
import torch
import torch.nn as nn
import math

In [19]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads):
        super().__init__()

        assert embed_dim % num_heads == 0, \
            "Embedding dimension must be divisible by number of heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Query, Key, Value projections
        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Final output projection
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        # Generate Q, K, V
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Split into multiple heads
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        )

        # Move head dimension forward
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Attention scores
        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        # Scale
        scores = scores / math.sqrt(self.head_dim)

        # Softmax
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Context vectors
        context = torch.matmul(
            attention_weights,
            V
        )

        # Restore dimensions
        context = context.transpose(1, 2)

        context = context.reshape(
            batch_size,
            seq_len,
            self.embed_dim
        )

        # Final projection
        output = self.fc_out(context)

        return output

In [20]:
batch_size = 1
seq_len = 3
embed_dim = 8
num_heads = 2

# Example input
x = torch.randn(
    batch_size,
    seq_len,
    embed_dim
)

mha = MultiHeadAttention(
    embed_dim=embed_dim,
    num_heads=num_heads
)

output = mha(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

print("\nOutput Tensor:\n")
print(output)

Input Shape : torch.Size([1, 3, 8])
Output Shape: torch.Size([1, 3, 8])

Output Tensor:

tensor([[[-1.1251e-01, -1.9139e-02,  2.5488e-01,  1.1417e-01, -4.7063e-01,
          -2.8006e-01,  3.7368e-03, -1.3729e-01],
         [-7.9498e-02, -1.0676e-02,  2.2506e-01,  1.3172e-01, -4.4479e-01,
          -2.5622e-01, -4.3282e-02, -9.4321e-02],
         [-6.1198e-02, -1.6851e-04,  2.1478e-01,  1.2806e-01, -4.3168e-01,
          -2.5884e-01, -3.9201e-02, -8.7415e-02]]], grad_fn=<ViewBackward0>)
